In [2]:
import sys
sys.path.append("/nfs/turbo/coe-venkvis/ziqiw-turbo/material_agent/GNoME_DREAMS_OER_screening/src/GNoME_DREAMS_OER_screening")

from ase.io import read, write
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.io.vasp.sets import Kpoints

from vasp.pre_defined_vasp_sets import (
    RPBE_relax_bulk_set, RPBE_relax_surface_set 
)

import os

In [3]:
# Example: Setting up a VASP calculation for bulk TiO2 (rutile)

# Load structure:
TiO2 = read("/nfs/turbo/coe-venkvis/ziqiw-turbo/material_agent/GNoME_DREAMS_OER_screening/notebooks/TiO2_rutile.cif")
TiO2_structure = AseAtomsAdaptor.get_structure(TiO2)

# Create VASP input set, using predefined relaxation settings for bulk 
# materials (RPBE_relax_bulk_set):
vasp_set = RPBE_relax_bulk_set(structure = TiO2_structure)


# Define output directory for VASP calculation:
calculation_dir = "./TiO2_VASP_calculation_test_1/"
os.makedirs(calculation_dir, exist_ok=True)

# Write VASP input files to the output directory.
# potcar_spec must be True, if you do not have acces to POTCAR files 
# locally (these are copyrighted by VASP)
vasp_set.write_input(output_dir = calculation_dir, potcar_spec=True) 

# You sould now have the following files in the calculation_dir:
# INCAR, KPOINTS, POSCAR, POTCAR.spec

In [4]:
# Example: Customizing INCAR and KPOINTS settings for a VASP calculation

# You can choose to modify the default settings by providing your own
# INCAR and KPOINTS settings when creating the VASP input set:
new_incar_settings = {
    "ENCUT": 10000,
    "EDIFF": 1e-6,
    "ISMEAR": 0,
    "SIGMA": 0.05,
}
k_points = Kpoints(kpts = [(3,3,4),])

calculation_dir = "./TiO2_VASP_calculation_test_2/"
os.makedirs(calculation_dir, exist_ok=True)

vasp_set = RPBE_relax_bulk_set(
    structure = TiO2_structure,
    user_incar_settings = new_incar_settings,
    user_kpoints_settings = k_points,
)

vasp_set.write_input(output_dir = calculation_dir, potcar_spec=True)

# You should now be able to see that your custom settings have been 
# applied in the INCAR and KPOINTS files in the calculation_dir.

In [5]:
# The actuall VASP calculation can now be run in the specified 
# directory via the run_vasp_via_custodian function. 

from vasp.vasp_calculation import (
    run_vasp_via_custodian, read_vasp_results, clean_up_vasp_directory
)


ModuleNotFoundError: No module named 'custodian'

In [ ]:
# The following line should be used to run the VASP calculation on the cluster<<<----- NOTE !!!
# Will not work without a proper VASP installation and license...
run_vasp_via_custodian(
    directory = calculation_dir,
#   handlers = [...], # You can specify custom custodian error handlers 
# - see https://materialsproject.github.io/custodian/custodian.vasp.handlers.html
    auto_npar = True, # Will choose optimal NPAR based on number of available cores
    max_errors = 5 # Maximum number of errors to tolerate before stopping the calculation
)

# After the calculation is complete, you can read the results:
atoms_list = read_vasp_results(
    directory = calculation_dir,
    return_only_final = True # Set to False if you want all intermediate structures
)

# Unnecessary files can be removed from the VASP calculation directory 
# by using the clean_up_vasp_directory function:
clean_up_vasp_directory(
    directory = calculation_dir,
)